In [ ]:
!pip install -q streamlit xgboost python-whois beautifulsoup4 requests
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 66.8 MB/s eta 0:00:00


In [ ]:
%%writefile features.py
import re, socket, ssl, requests
from datetime import datetime
from urllib.parse import urlparse
from bs4 import BeautifulSoup

try:
    import whois
except Exception:
    whois = None

SHORT = r"bit\.ly|goo\.gl|tinyurl|t\.co|ow\.ly|is\.gd|buff\.ly|adf\.ly|cutt\.ly|rebrand\.ly|tiny\.cc"

def _pct(p, lo, hi):
    return 1 if p < lo else (0 if p <= hi else -1)

def _date(d):
    if isinstance(d, list):
        d = d[0] if d else None
    if isinstance(d, datetime):
        return d.replace(tzinfo=None)
    return None

def _ssl_ok(host):
    try:
        ctx = ssl.create_default_context()
        with socket.create_connection((host, 443), timeout=5) as s:
            with ctx.wrap_socket(s, server_hostname=host):
                return True
    except Exception:
        return False

def extract_features(url):
    url = url.strip()
    if not url.startswith(("http://", "https://")):
        url = "http://" + url
    p = urlparse(url)
    host = (p.hostname or "").lower()
    if not host:
        raise ValueError("Invalid URL")
    domain = host[4:] if host.startswith("www.") else host
    try:
        port = p.port
    except ValueError:
        port = -1

    f = {}

    # ---------- URL based ----------
    f["having_IP_Address"] = -1 if re.fullmatch(r"(\d{1,3}\.){3}\d{1,3}", host) else 1
    n = len(url)
    f["URL_Length"] = 1 if n < 54 else (0 if n <= 75 else -1)
    f["Shortining_Service"] = -1 if re.search(SHORT, url) else 1
    f["having_At_Symbol"] = -1 if "@" in url else 1
    f["double_slash_redirecting"] = -1 if url.rfind("//") > 7 else 1
    f["Prefix_Suffix"] = -1 if "-" in host else 1
    dots = domain.count(".")
    f["having_Sub_Domain"] = 1 if dots <= 1 else (0 if dots == 2 else -1)
    f["port"] = -1 if port not in (None, 80, 443) else 1
    f["HTTPS_token"] = -1 if "https" in host else 1

    # ---------- WHOIS / DNS ----------
    created = expires = None
    if whois is not None:
        try:
            w = whois.whois(domain)
            created = _date(w.creation_date)
            expires = _date(w.expiration_date)
        except Exception:
            pass
    now = datetime.now()
    age_days = (now - created).days if created else 0

    f["Domain_registeration_length"] = 1 if expires and (expires - now).days > 365 else -1
    f["age_of_domain"] = 1 if created and age_days >= 180 else -1
    f["Abnormal_URL"] = 1 if created else -1
    try:
        socket.gethostbyname(host)
        f["DNSRecord"] = 1
    except Exception:
        f["DNSRecord"] = -1

    # ---------- SSL (stricter) ----------
    if p.scheme == "https":
        ok = _ssl_ok(host)
        f["SSLfinal_State"] = 1 if (ok and age_days >= 365) else (0 if ok else -1)
    else:
        f["SSLfinal_State"] = -1

    # ---------- Page content based ----------
    html, hist = "", 0
    try:
        r = requests.get(url, timeout=8, headers={"User-Agent": "Mozilla/5.0"})
        html, hist = r.text, len(r.history)
    except Exception:
        pass
    fetched = bool(html)
    soup = BeautifulSoup(html, "html.parser")

    def ext(link):
        if not link or link.startswith(("#", "javascript", "mailto")):
            return False
        nl = urlparse(link).netloc
        return bool(nl) and domain not in nl

    imgs = [t.get("src") or t.get("data-src") for t in soup.find_all(["img", "audio", "embed", "iframe"])]
    imgs = [i for i in imgs if i]
    f["Request_URL"] = _pct(sum(map(ext, imgs)) / len(imgs) * 100, 22, 61) if imgs else 1

    anchors = [a.get("href") for a in soup.find_all("a")]
    anchors = [a for a in anchors if a is not None]
    bad = [a for a in anchors if a.startswith(("#", "javascript", "mailto")) or ext(a)]
    f["URL_of_Anchor"] = _pct(len(bad) / len(anchors) * 100, 31, 67) if anchors else 1

    tags = [t.get("href") or t.get("src") for t in soup.find_all(["meta", "script", "link"])]
    tags = [t for t in tags if t]
    f["Links_in_tags"] = _pct(sum(map(ext, tags)) / len(tags) * 100, 17, 81) if tags else 1

    sfh = 1
    for form in soup.find_all("form"):
        act = form.get("action", "") or ""
        if act in ("", "about:blank"):
            sfh = -1
            break
        if ext(act):
            sfh = 0
    f["SFH"] = sfh

    f["Submitting_to_email"] = -1 if re.search(r"mailto:|mail\(", html) else 1
    f["Redirect"] = 1 if hist <= 1 else (0 if hist < 5 else -1)
    f["on_mouseover"] = -1 if re.search(r"onmouseover\s*=\s*[\"'][^\"']*window\.status", html, re.I) else 1
    f["RightClick"] = -1 if re.search(r"event\.button\s*==\s*2|contextmenu", html, re.I) else 1
    f["popUpWidnow"] = -1 if re.search(r"window\.open\(|alert\(", html) else 1
    f["Iframe"] = -1 if soup.find("iframe") else 1

    icon = soup.find("link", rel=lambda v: v and "icon" in str(v).lower())
    f["Favicon"] = -1 if icon and ext(icon.get("href")) else 1

    # Page open aagala na suspicious
    if not fetched:
        for k in ["Request_URL", "URL_of_Anchor", "Links_in_tags", "SFH"]:
            f[k] = -1

    # extra info (model-ku pogaadhu, app-la use aagum)
    f["_fetched"] = fetched
    f["_age_days"] = age_days
    return f

Writing features.py


In [ ]:
import pandas as pd, numpy as np, joblib
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             confusion_matrix, roc_curve, roc_auc_score)
from xgboost import XGBClassifier
ALL_FEATURES = [
 "having_IP_Address","URL_Length","Shortining_Service","having_At_Symbol",
 "double_slash_redirecting","Prefix_Suffix","having_Sub_Domain","SSLfinal_State",
 "Domain_registeration_length","Favicon","port","HTTPS_token","Request_URL",
 "URL_of_Anchor","Links_in_tags","SFH","Submitting_to_email","Abnormal_URL",
 "Redirect","on_mouseover","RightClick","popUpWidnow","Iframe","age_of_domain",
 "DNSRecord","web_traffic","Page_Rank","Google_Index","Links_pointing_to_page",
 "Statistical_report"
]
# These need paid APIs, so they are not used for live prediction
DROP = ["web_traffic","Page_Rank","Google_Index","Links_pointing_to_page","Statistical_report"]

url = "https://raw.githubusercontent.com/npapernot/phishing-detection/master/dataset.csv"
df = pd.read_csv(url, header=None)
df = df.iloc[:, -31:]                      # 30 features + Result
df.columns = ALL_FEATURES + ["Result"]
df["Result"] = df["Result"].map({-1: 1, 1: 0})   # 1 = Phishing, 0 = Legitimate

feature_names = [c for c in ALL_FEATURES if c not in DROP]
X = df[feature_names]
y = df["Result"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

xgb_model = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                          subsample=0.8, colsample_bytree=0.8,
                          eval_metric="logloss", random_state=42)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
print(f"Accuracy: {accuracy*100:.2f}%  Precision: {precision*100:.2f}%  Recall: {recall*100:.2f}%  AUC: {auc:.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges",
            xticklabels=["Legitimate","Phishing"], yticklabels=["Legitimate","Phishing"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix - XGBoost")
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=120); plt.close()

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"XGBoost (AUC = {auc:.3f})", color="#DD8452")
plt.plot([0,1],[0,1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve"); plt.legend()
plt.tight_layout(); plt.savefig("roc_curve.png", dpi=120); plt.close()

# Feature importance
imp = pd.Series(xgb_model.feature_importances_, index=feature_names).sort_values(ascending=False)
plt.figure(figsize=(8,8))
imp.head(15).plot(kind="barh", color="#DD8452")
plt.gca().invert_yaxis(); plt.title("Top 15 Important Features"); plt.xlabel("Importance Score")
plt.tight_layout(); plt.savefig("feature_importance.png", dpi=120); plt.close()

joblib.dump(xgb_model, "phishing_xgboost_model.pkl")
joblib.dump(feature_names, "feature_names.pkl")
joblib.dump({"accuracy": accuracy, "precision": precision, "recall": recall, "auc": auc}, "metrics.pkl")
print("Model and all files saved successfully")

Accuracy: 94.98%  Precision: 95.40%  Recall: 93.16%  AUC: 0.9914
Model and all files saved successfully


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
from features import extract_features

st.set_page_config(page_title="Phishing Website Detector", page_icon="🛡️", layout="wide")

model = joblib.load("phishing_xgboost_model.pkl")
feature_names = joblib.load("feature_names.pkl")
metrics = joblib.load("metrics.pkl")

st.title("Phishing Website Detection System")
st.markdown("**Algorithm:** XGBoost Classifier + Rule-based Red Flag Check")

tab1, tab2 = st.tabs(["Predict", "Model Performance"])

with tab1:
    st.subheader("Enter a Website URL")
    url_input = st.text_input("Website URL", placeholder="https://example.com")

    if st.button("Check Website", use_container_width=True, type="primary"):
        if not url_input.strip():
            st.warning("Please enter a URL")
        else:
            try:
                with st.spinner("Analysing website..."):
                    feats = extract_features(url_input)
                    input_df = pd.DataFrame([feats])[feature_names]
                    pred = int(model.predict(input_df)[0])
                    prob = float(model.predict_proba(input_df)[0][1])  # phishing probability

                flags = []
                if feats["having_IP_Address"] == -1: flags.append("URL-la IP address irukku")
                if feats["having_At_Symbol"] == -1: flags.append("URL-la @ symbol irukku")
                if feats["Shortining_Service"] == -1: flags.append("URL shortener use panniruku")
                if feats["Prefix_Suffix"] == -1: flags.append("Domain-la '-' irukku")
                if feats["SSLfinal_State"] == -1: flags.append("HTTPS illa / SSL problem")
                if feats["age_of_domain"] == -1: flags.append("Domain romba pudhusu / WHOIS info illa")
                if feats["DNSRecord"] == -1: flags.append("DNS record illa")
                if not feats["_fetched"]: flags.append("Page open aagala (site dead / blocked)")

                if pred == 1:
                    st.error(f"PHISHING WEBSITE DETECTED - Confidence: {prob*100:.2f}%")
                elif len(flags) >= 3:
                    st.warning(f"SUSPICIOUS - Model legitimate-nu sonnalum {len(flags)} red flags irukku")
                else:
                    st.success(f"LEGITIMATE WEBSITE - Confidence: {(1-prob)*100:.2f}%")
                st.progress(prob)

                if flags:
                    st.write("**Red flags:**")
                    for fl in flags:
                        st.write("- " + fl)

                with st.expander("Extracted features"):
                    feat_df = pd.DataFrame({
                        "Feature": feature_names,
                        "Value": [int(input_df.iloc[0][c]) for c in feature_names]
                    })
                    st.table(feat_df)
            except Exception as e:
                st.error(f"Could not analyse this URL: {e}")

with tab2:
    st.subheader("Model Performance Metrics")
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Accuracy", f"{metrics['accuracy']*100:.2f}%")
    c2.metric("Precision", f"{metrics['precision']*100:.2f}%")
    c3.metric("Recall", f"{metrics['recall']*100:.2f}%")
    c4.metric("AUC Score", f"{metrics['auc']:.4f}")

    colA, colB = st.columns(2)
    with colA:
        st.image("confusion_matrix.png", caption="Confusion Matrix")
    with colB:
        st.image("roc_curve.png", caption="ROC Curve")
    st.image("feature_importance.png", caption="Top 15 Important Features")

Writing app.py


In [ ]:
import time, re, subprocess, os

# stop old instances if you re-run
os.system("pkill -f streamlit; pkill -f cloudflared")
time.sleep(2)

subprocess.Popen("streamlit run app.py --server.port 8501 --server.headless true &> /content/logs.txt",
                 shell=True)
time.sleep(8)
subprocess.Popen("./cloudflared tunnel --url http://localhost:8501 --logfile /content/cf_logs.txt --loglevel info",
                 shell=True)

link = None
for _ in range(30):
    time.sleep(2)
    try:
        with open("/content/cf_logs.txt") as f:
            m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", f.read())
        if m:
            link = m.group(0)
            break
    except FileNotFoundError:
        pass

print("APP LINK:", link if link else "Not ready, run this cell again")

APP LINK: https://moments-smell-pvc-visibility.trycloudflare.com
